# 04 数据库速查
直接用 SQL 看 `bars` 表。

In [ ]:
import pandas as pd, sqlalchemy as sa
from ibkr_quant.db import engine
def q(sql, **params):
    return pd.read_sql(sa.text(sql), engine, params=params)

In [ ]:
q('select symbol, bar_size, count(*) as rows, min(ts)::date as first, max(ts)::date as last from bars group by 1,2 order by 1')

In [ ]:
q("select ts::date as date, open, high, low, close, volume from bars where symbol = :sym order by ts desc limit 10", sym='SPY')

In [ ]:
# 单日最大涨跌
q("""select * from (
         select symbol, ts::date as date,
                round((close / lag(close) over (partition by symbol order by ts) - 1)::numeric * 100, 2) as ret_pct
         from bars where bar_size = '1 day') t
       where ret_pct is not null order by abs(ret_pct) desc limit 10""")